# NGS QC Data Analytics Pipeline

# NGS QC Data Extraction Pipeline

This notebook extracts quality control (QC) metrics from PDF reports and prepares a structured dataframe for downstream analysis and visualization.

## Steps:
1. Load PDF files
2. Extract relevant QC metrics
3. Clean and standardize data
4. Export dataframe for dashboard use

In [ ]:
!pip install OpenAI
!pip install pypdf
!pip install -q langchain langchain-community openai chromadb pypdf



# Using Langchain to Load Documents

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:

import os
from langchain_community.document_loaders import PyPDFLoader
from google.colab import files

# 1) Upload PDF files
uploaded = files.upload()  # pick one or more PDF reports
pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
if not pdf_paths:
    raise SystemExit("No PDFs uploaded.")

# 2) Load ALL pages from each PDF
all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    pages = loader.load_and_split()   # load all pages (no truncation)
    all_docs.append({"path": path, "pages": pages})

# 3) Quick preview of the first 600 characters per document
for doc in all_docs:
    print(f"\n=== {doc['path']} — {len(doc['pages'])} pages ===")
    if doc["pages"]:
        print(doc["pages"][0].page_content[:600])

# 4) Flatten into one list for embedding or text search
all_pages = []
for doc in all_docs:
    for i, page in enumerate(doc["pages"], start=1):
        page.metadata["source_file"] = doc["path"]
        page.metadata["page_number"] = i
        all_pages.append(page)

print(f"\nLoaded {len(all_docs)} PDFs with {len(all_pages)} total pages.")


# Parsing the data

In [ ]:
import re
import pandas as pd
from dateutil import parser as dtparser

# ---------------------------
# Helpers
# ---------------------------

def _to_number(val):
    """
    Convert extracted string to int/float when possible; otherwise return stripped string.
    NOTE: For lot numbers that may include letters (e.g., 3000459A), this will keep them as text.
    """
    if val is None:
        return None
    v = str(val).replace(",", "").strip()
    v = re.sub(r"\s+", " ", v)

    # If it contains any letters, keep as string
    if re.search(r"[A-Za-z]", v):
        return v

    try:
        if re.fullmatch(r"-?\d+", v):
            return int(v)
        if re.fullmatch(r"-?\d+\.\d+", v):
            return float(v)
    except Exception:
        pass

    return v

def _parse_datetime_fuzzy(raw):
    """Fuzzy parse common Ion Torrent date/time strings into pandas Timestamp."""
    if not raw:
        return None
    s = str(raw).strip()

    # Normalize known variants
    s = s.replace("Sept.", "Sep.").replace("Sept", "Sep")
    s = s.replace("a.m.", "AM").replace("p.m.", "PM").replace("a. m.", "AM").replace("p. m.", "PM")
    s = re.sub(r"\s+", " ", s)

    try:
        return pd.to_datetime(dtparser.parse(s, fuzzy=True))
    except Exception:
        return None

def extract_analysis_run_date(text):
    """
    Anchor on 'Analysis Details' and extract a date string.
    Prefer 'Analysis Date' if present, else fall back to 'Run Date'.
    Returns: (raw_string, parsed_timestamp)
    """
    m = re.search(r"Analysis\s+Details", text, re.IGNORECASE)
    if not m:
        return (None, None)

    start = m.start()
    window = text[start : start + 12000]  # safer than 4000 for messy PDF extraction

    # Prefer Analysis Date
    m2 = re.search(r"Analysis\s*Date\s*[:\-]?\s*(.+?)(?:\n|$)", window, re.IGNORECASE)
    # Fall back to Run Date
    if not m2:
        m2 = re.search(r"Run\s*Date\s*[:\-]?\s*(.+?)(?:\n|$)", window, re.IGNORECASE)
    # Last fallback if newlines collapsed
    if not m2:
        m2 = re.search(r"(?:Analysis\s*Date|Run\s*Date)\s*[:\-]?\s*(.{10,80})", window, re.IGNORECASE)

    if not m2:
        return (None, None)

    raw = re.sub(r"\s+", " ", m2.group(1)).strip()
    parsed = _parse_datetime_fuzzy(raw)
    return (raw, parsed)

# ---------------------------
# Parser (SAFE lot fallback)
# ---------------------------

def parse_ngs_metrics(text, source_file):
    data = {"Source_File": source_file}

    # Base patterns (same idea as your original)
    patterns = {
        "Run_Name": r"Run Report for\s+(.+?)(?:\n|$)",
        "ISP_Loading_%": r"Key Signal\s*\n\s*([\d.]+)\s*%",

        # NOTE: keep Total_Reads here if you want, but we'll override it below with a safer extractor
        "Total_Reads": r"Total Reads\s*\n\s*([\d,]+)",

        "Instrument": r"Instrument\s+([\w\-]+)",
        "Chip_Barcode": r"Chip Barcode\s+(\w+)",
        "Chip_Lot_Number": r"Chip Lot Number\s+(\w+)",
        "Chip_Wafer": r"Chip Wafer\s+(\d+)",

        # S5 reagent lines (present in some reports)
        # Use (\S+) not (\d+) to allow alphanumeric lots
        "Cleaning_Solution_Lot": r"Ion S5 Cleaning Solution\s+\S+\s+(\S+)",
        "ExT_Sequencing_Reagent_Lot": r"Ion S5 ExT Sequencing Reagent\s+\S+\s+(\S+)",
        "ExT_Wash_Solution_Lot": r"Ion S5 ExT Wash Solution\s+\S+\s+(\S+)",
    }

    # 1) Apply base patterns FIRST (prevents missing columns)
    for key, pattern in patterns.items():
        m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        data[key] = _to_number(m.group(1)) if m else None

    # ---- Total_Reads extraction (handles both layouts) ----
    # Some PDF extractions put the number ABOVE the label:
    #   21,161,316
    #   Total Reads
    # and sometimes put the label ABOVE the number:
    #   Total Reads
    #   21,161,316
    #
    # We also enforce "at least 4 digits / comma-grouping" to avoid accidentally grabbing percentages.
    tr_a = re.search(r"Total\s+Reads\s*\n\s*([\d,]{4,})\b", text, re.IGNORECASE)              # label then number
    tr_b = re.search(r"\b([\d,]{4,})\s*\n\s*Total\s+Reads\b", text, re.IGNORECASE)          # number then label
    tr_c = re.search(r"Total\s+Reads\s*[:\-]?\s*([\d,]{4,})\b", text, re.IGNORECASE)        # same-line fallback

    if tr_a:
        data["Total_Reads"] = _to_number(tr_a.group(1))
    elif tr_b:
        data["Total_Reads"] = _to_number(tr_b.group(1))
    elif tr_c:
        data["Total_Reads"] = _to_number(tr_c.group(1))
    # else: keep whatever the base pattern got (or None)

    # 2) ALWAYS use Analysis Details date (raw + parsed)
    raw_date, parsed_date = extract_analysis_run_date(text)
    data["Run_Date_raw"] = raw_date
    data["Run_Date"] = parsed_date

    # 3) Usable Reads % (dual layout)
    usable_a = re.search(r"([\d.]+)\s*%\s*\n\s*Usable Reads", text, re.IGNORECASE)
    usable_b = re.search(r"Usable Reads\s*\n\s*([\d.]+)\s*%", text, re.IGNORECASE)
    if usable_a:
        data["Usable_Reads_%"] = float(usable_a.group(1))
    elif usable_b:
        data["Usable_Reads_%"] = float(usable_b.group(1))
    else:
        data["Usable_Reads_%"] = None

    # 4) Read length triplet (Mean / Median / Mode)
    match_lengths = re.search(
        r"([\d.,]+)\s*bp\s+([\d.,]+)\s*bp\s+([\d.,]+)\s*bp\s*\n\s*Mean\s+Median\s+Mode",
        text,
        re.IGNORECASE
    )
    if match_lengths:
        data["Mean_Read_Length_bp"] = float(match_lengths.group(1).replace(",", ""))
        data["Median_Read_Length_bp"] = float(match_lengths.group(2).replace(",", ""))
        data["Mode_bp"] = float(match_lengths.group(3).replace(",", ""))
    else:
        data["Mean_Read_Length_bp"] = None
        data["Median_Read_Length_bp"] = None
        data["Mode_bp"] = None

    # 5) Chef Summary fallbacks (do NOT overwrite other fields)
    chef_reagent = re.search(r"Reagent Lot Number\s+([A-Za-z0-9]+)", text, re.IGNORECASE)
    chef_solution = re.search(r"Solution Lot Number\s+([A-Za-z0-9]+)", text, re.IGNORECASE)

    data["Chef_Reagent_Lot"] = chef_reagent.group(1) if chef_reagent else None
    data["Chef_Solution_Lot"] = chef_solution.group(1) if chef_solution else None

    # Fill only if missing
    if data["ExT_Sequencing_Reagent_Lot"] is None and data["Chef_Reagent_Lot"] is not None:
        data["ExT_Sequencing_Reagent_Lot"] = data["Chef_Reagent_Lot"]

    if data["ExT_Wash_Solution_Lot"] is None and data["Chef_Solution_Lot"] is not None:
        data["ExT_Wash_Solution_Lot"] = data["Chef_Solution_Lot"]

    if data["Cleaning_Solution_Lot"] is None and data["Chef_Solution_Lot"] is not None:
        data["Cleaning_Solution_Lot"] = data["Chef_Solution_Lot"]

    return data


# ---------------------------
# COMBINE & PROCESS
# ---------------------------

records = []
for doc in all_docs:
    full_text = "\n".join([p.page_content for p in doc["pages"]])
    records.append(parse_ngs_metrics(full_text, doc["path"]))

df_metrics = pd.DataFrame(records)

# Ensure dtype
df_metrics["Run_Date"] = pd.to_datetime(df_metrics["Run_Date"], errors="coerce")

# Sort by date
df_metrics = df_metrics.sort_values("Run_Date", na_position="last")

print("✅ Extracted metrics:\n")
display(df_metrics)

df_metrics.to_csv("ngs_run_metrics.csv", index=False)
print("💾 Saved as ngs_run_metrics.csv")

# Coverage report
missing = df_metrics["Run_Date"].isna().sum()
print(f"\n📌 Run_Date missing: {missing} / {len(df_metrics)}")

lot_cols = ["Cleaning_Solution_Lot", "ExT_Sequencing_Reagent_Lot", "ExT_Wash_Solution_Lot"]
for c in lot_cols:
    if c in df_metrics.columns:
        print(f"📌 Missing {c}: {df_metrics[c].isna().sum()} / {len(df_metrics)}")

# Show a few examples if lots missing
if any(df_metrics[c].isna().any() for c in lot_cols if c in df_metrics.columns):
    display(df_metrics[df_metrics[lot_cols].isna().any(axis=1)][
        ["Source_File", "Cleaning_Solution_Lot", "ExT_Sequencing_Reagent_Lot", "ExT_Wash_Solution_Lot",
         "Chef_Reagent_Lot", "Chef_Solution_Lot"]
    ].head(5))


# Some verifications before final analysis

In [ ]:
df_metrics_sorted = df_metrics.sort_values("Run_Date")


In [ ]:
summary = (
    pd.DataFrame({
        "column": df_metrics.columns,
        "total_rows": len(df_metrics),
        "missing_count": df_metrics.isna().sum().values,
    })
)

summary["missing_pct"] = (
    summary["missing_count"] / summary["total_rows"] * 100
).round(1)

display(summary.sort_values("missing_pct", ascending=False))


# NEW ANALYSIS - Hampel / MAD z-score

In [ ]:
# ---------------------------
# Hampel/MAD QC (multi-metric) + Export + Download (Colab)
# ---------------------------

import numpy as np
import pandas as pd
from google.colab import files

def add_hampel_mad_columns_multi(
    df: pd.DataFrame,
    date_col: str = "Run_Date",
    value_cols=("Mean_Read_Length_bp",),
    group_cols=("Instrument",),
    window: int = 15,
    k: float = 3.5,
    min_periods: int = 8,
    drop_dupes: bool = True,
    outlier_na=pd.NA,
):
    d = df.copy()

    # Parse date
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")

    # Normalize inputs
    value_cols = list(value_cols)
    group_cols = list(group_cols) if group_cols else []

    # Validate required columns
    for m in value_cols:
        if m not in d.columns:
            raise KeyError(f"Missing metric column: {m}")
        d[m] = pd.to_numeric(d[m], errors="coerce")

    for gc in group_cols:
        if gc not in d.columns:
            raise KeyError(f"Missing group column: {gc}")

    # Sort by group + date
    sort_cols = group_cols + [date_col] if group_cols else [date_col]
    d = d.sort_values(sort_cols)

    # Optional: drop duplicates per group/date
    if drop_dupes and group_cols:
        d = d.drop_duplicates(subset=group_cols + [date_col], keep="last")

    def _compute(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values(date_col).copy()

        for m in value_cols:
            rm = g[m].rolling(window=window, min_periods=min_periods).median()
            mad = (g[m] - rm).abs().rolling(window=window, min_periods=min_periods).median()
            robust_sigma = 1.4826 * mad
            robust_z = (g[m] - rm) / robust_sigma

            g[f"robust_median__{m}"] = rm
            g[f"mad__{m}"] = mad
            g[f"robust_sigma__{m}"] = robust_sigma
            g[f"robust_z__{m}"] = robust_z

            g[f"H_outlier__{m}"] = np.where(
                robust_z.isna(),
                outlier_na,
                robust_z.abs() > k
            )
        return g

    if group_cols:
        def _compute_with_keys(keys, g):
            if not isinstance(keys, tuple):
                keys = (keys,)
            for gc, kv in zip(group_cols, keys):
                g[gc] = kv
            return _compute(g)

        parts = []
        for keys, g in d.groupby(group_cols, sort=False):
            parts.append(_compute_with_keys(keys, g))

        out = pd.concat(parts, ignore_index=True)
        out = out.sort_values(group_cols + [date_col]).reset_index(drop=True)
        return out

    out = _compute(d).reset_index(drop=True)
    return out


# ---------------------------
# Load metrics dataframe (if not already in memory)
# ---------------------------
df_metrics = pd.read_csv("/content/ngs_run_metrics.csv")

# Ensure date dtype
df_metrics["Run_Date"] = pd.to_datetime(df_metrics["Run_Date"], errors="coerce")

# ---------------------------
# 1) Define which QC metric columns to compute robust baselines for
#    (Replace this list with explicit list if preferred.)
# ---------------------------

# Include ONLY these metrics (explicit list)
metrics_to_include = [
    "Total_Reads",
    "Usable_Reads_%",
    "Mean_Read_Length_bp",
    "Median_Read_Length_bp",
    "Mode_bp",
]

# Keep only metrics in df_metrics (avoids KeyError if a column is missing)
metrics_to_include = [m for m in metrics_to_include if m in df_metrics.columns]

# Ensure numeric coercion (in case parser produced strings)
for m in metrics_to_include:
    df_metrics[m] = pd.to_numeric(df_metrics[m], errors="coerce")

# ---------------------------
# 2) Run Hampel/MAD per grouping choice
#    You said you want to evaluate over time behavior per:
#      - Instrument
#      - Chip_Lot_Number
#      - Cleaning_Solution_Lot
#      - ExT_Sequencing_Reagent_Lot
#      - ExT_Wash_Solution_Lot
#    We'll run 5 separate analyses and export 5 CSVs.
# ---------------------------

grouping_options = [
    ("Instrument",),
    ("Chip_Lot_Number",),
    ("Cleaning_Solution_Lot",),
    ("ExT_Sequencing_Reagent_Lot",),
    ("ExT_Wash_Solution_Lot",),
]

for group_cols in grouping_options:
    # Skip grouping if the grouping column doesn't exist in df_metrics
    if any(gc not in df_metrics.columns for gc in group_cols):
        print(f"⚠️ Skipping group {group_cols} because column not found.")
        continue

    df_hampel = add_hampel_mad_columns_multi(
        df=df_metrics,
        date_col="Run_Date",
        value_cols=tuple(metrics_to_include),
        group_cols=group_cols,
        window=15,
        k=3.5,
        min_periods=8,
        drop_dupes=False,  # better for lots: you may have multiple runs per date
    )

    out_path = f"/content/ngs_qc_hampel_mad_results__by_{group_cols[0]}.csv"
    df_hampel.to_csv(out_path, index=False)
    print(f"💾 Saved: {out_path}")

    # Download each file
    files.download(out_path)


# Validation to confirm everything is working well

In [ ]:
df_hampel = add_hampel_mad_columns_multi(
    df_metrics,
    date_col="Run_Date",
    value_cols=[
    "Mean_Read_Length_bp",
    "Median_Read_Length_bp",
    "Mode_bp",
    "Total_Reads",
    "Usable_Reads_%",
]
,
    group_cols=("Instrument",),
    window=15,
    k=3.5,
    min_periods=8,
)

df_hampel.columns.tolist()


# Visualizations

In [ ]:
# ----------------------------
# CHUNK 2 (FULL) — compute + plot per instrument with a fixed Y-axis
# (Option 2: per-metric robust columns)
# Y-axis range includes BOTH the data points and the Hampel band
# ----------------------------

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ============================================================
# Helper: Option-2-compatible plotting function
# ============================================================
def plot_hampel_timeseries_option2(
    df: pd.DataFrame,
    date_col: str,
    value_col: str,
    group_values: dict | None = None,
    k: float = 3.5,
    title: str = "",
):
    """
    Plot Hampel/MAD longitudinal QC for Option 2 outputs.

    Requires:
      robust_median__<metric>
      robust_sigma__<metric>
    Optional:
      robust_z__<metric>
      H_outlier__<metric>
    """

    d = df.copy()

    # Apply group filters
    if group_values:
        for c, v in group_values.items():
            if c in d.columns:
                d = d[d[c].astype(str) == str(v)].copy()

    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.sort_values(date_col)

    med_col = f"robust_median__{value_col}"
    sig_col = f"robust_sigma__{value_col}"
    z_col   = f"robust_z__{value_col}"
    out_col = f"H_outlier__{value_col}"

    median = pd.to_numeric(d[med_col], errors="coerce")
    sigma  = pd.to_numeric(d[sig_col], errors="coerce")

    band_lo = median - k * sigma
    band_hi = median + k * sigma

    # Outlier mask
    if out_col in d.columns:
        out_mask = d[out_col].fillna(False).astype(bool)
    elif z_col in d.columns:
        out_mask = pd.to_numeric(d[z_col], errors="coerce").abs() > k
    else:
        out_mask = pd.Series(False, index=d.index)

    normal = d[~out_mask]
    outl = d[out_mask]

    fig = go.Figure()

    # Hampel band
    fig.add_trace(go.Scatter(
        x=d[date_col], y=band_lo,
        mode="lines", line=dict(width=0),
        hoverinfo="skip", showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=d[date_col], y=band_hi,
        mode="lines", line=dict(width=0),
        fill="tonexty",
        name=f"Robust band (±{k}σ)",
        opacity=0.25,
        hoverinfo="skip"
    ))

    # Rolling median
    fig.add_trace(go.Scatter(
        x=d[date_col], y=median,
        mode="lines",
        name="Rolling robust median"
    ))

    # Normal points
    fig.add_trace(go.Scatter(
        x=normal[date_col],
        y=pd.to_numeric(normal[value_col], errors="coerce"),
        mode="markers",
        name="Runs (normal)",
        customdata=pd.to_numeric(normal[z_col], errors="coerce") if z_col in normal.columns else None,
        hovertemplate=(
            f"{date_col}: %{{x}}<br>"
            f"{value_col}: %{{y}}"
            + ("<br>robust_z: %{customdata:.2f}" if z_col in normal.columns else "")
            + "<extra></extra>"
        )
    ))

    # Outliers
    if len(outl):
        fig.add_trace(go.Scatter(
            x=outl[date_col],
            y=pd.to_numeric(outl[value_col], errors="coerce"),
            mode="markers",
            name="Runs (outlier)",
            marker=dict(size=10, symbol="x"),
            customdata=pd.to_numeric(outl[z_col], errors="coerce") if z_col in outl.columns else None,
            hovertemplate=(
                f"{date_col}: %{{x}}<br>"
                f"{value_col}: %{{y}}"
                + ("<br>robust_z: %{customdata:.2f}" if z_col in outl.columns else "")
                + "<extra></extra>"
            )
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Run date",
        yaxis_title=value_col,
        hovermode="x unified",
        height=520,
        margin=dict(l=10, r=10, t=40, b=10),
    )

    return fig


# ------------------------------------------------------------
# Parameters (keep consistent between computation and plotting)
# ------------------------------------------------------------
k_plot = 3.5
window = 15
min_periods = 8

# Metric to plot
metric = "Mean_Read_Length_bp"

# ------------------------------------------------------------
# Compute Hampel/MAD columns (Option 2)
# ------------------------------------------------------------
df_hampel_len = add_hampel_mad_columns_multi(
    df_metrics,
    date_col="Run_Date",
    value_cols=[metric],
    group_cols=("Instrument",),
    window=window,
    k=k_plot,
    min_periods=min_periods,
)

# Metric-specific column names
med_col = f"robust_median__{metric}"
sig_col = f"robust_sigma__{metric}"

# ------------------------------------------------------------
# Compute band limits for global Y-axis
# ------------------------------------------------------------
df_hampel_len["band_lo"] = (
    pd.to_numeric(df_hampel_len[med_col], errors="coerce")
    - k_plot * pd.to_numeric(df_hampel_len[sig_col], errors="coerce")
)

df_hampel_len["band_hi"] = (
    pd.to_numeric(df_hampel_len[med_col], errors="coerce")
    + k_plot * pd.to_numeric(df_hampel_len[sig_col], errors="coerce")
)

y_min = np.nanmin([
    pd.to_numeric(df_hampel_len[metric], errors="coerce").min(),
    pd.to_numeric(df_hampel_len["band_lo"], errors="coerce").min(),
])

y_max = np.nanmax([
    pd.to_numeric(df_hampel_len[metric], errors="coerce").max(),
    pd.to_numeric(df_hampel_len["band_hi"], errors="coerce").max(),
])

padding = 0.05 * (y_max - y_min) if pd.notna(y_min) and pd.notna(y_max) and y_max > y_min else 0
y_range = [float(y_min - padding), float(y_max + padding)]

# ------------------------------------------------------------
# Plot per instrument with identical Y-axis
# ------------------------------------------------------------
for inst in sorted(df_hampel_len["Instrument"].dropna().astype(str).unique()):
    fig = plot_hampel_timeseries_option2(
        df=df_hampel_len,
        date_col="Run_Date",
        value_col=metric,
        group_values={"Instrument": inst},
        k=k_plot,
        title=f"{metric.replace('_', ' ')} over time — {inst} (Hampel/MAD ±{k_plot}σ)",
    )
    fig.update_yaxes(range=y_range)
    fig.show()


In [ ]:
import plotly.express as px

# ----------------------------
# CONFIG
# ----------------------------
LOT_COL = "Chip_Lot_Number"
METRIC_COL = "Mean_Read_Length_bp"

# ----------------------------
# Prepare data (minimal, safe filtering)
# ----------------------------
df_box = df_hampel_len[
    df_hampel_len[LOT_COL].notna() &
    df_hampel_len[METRIC_COL].notna() &
    df_hampel_len["Instrument"].notna()
].copy()

# Ensure lot numbers are treated as categorical (not numeric sorting)
df_box[LOT_COL] = df_box[LOT_COL].astype(str)

# ----------------------------
# Order chip lots by median read length (VERY helpful visually)
# ----------------------------
lot_order = (
    df_box.groupby(LOT_COL)[METRIC_COL]
    .median()
    .sort_values()
    .index
)

# ----------------------------
# Build box plot
# ----------------------------
fig = px.box(
    df_box,
    x=LOT_COL,
    y=METRIC_COL,
    color="Instrument",
    boxmode="group",       # side-by-side boxes per lot
    points="outliers",     # cleaner; change to "all" if N is small
)

fig.update_layout(
    title="Mean Read Length (bp) by Chip Lot Number (colored by Instrument)",
    xaxis_title="Chip Lot Number",
    yaxis_title="Mean Read Length (bp)",
    legend_title="Instrument",
    height=550
)

# Apply ordered lots
fig.update_xaxes(
    categoryorder="array",
    categoryarray=lot_order
)

fig.show()


In [ ]:
import plotly.express as px

# ----------------------------
# CONFIG
# ----------------------------
LOT_COL = "Cleaning_Solution_Lot"
METRIC_COL = "Mean_Read_Length_bp"

# ----------------------------
# Prepare data (minimal, safe filtering)
# ----------------------------
df_box = df_hampel_len[
    df_hampel_len[LOT_COL].notna() &
    df_hampel_len[METRIC_COL].notna() &
    df_hampel_len["Instrument"].notna()
].copy()

# Ensure lot numbers are treated as categorical (not numeric sorting)
df_box[LOT_COL] = df_box[LOT_COL].astype(str)

# ----------------------------
# Order chip lots by median read length (VERY helpful visually)
# ----------------------------
lot_order = (
    df_box.groupby(LOT_COL)[METRIC_COL]
    .median()
    .sort_values()
    .index
)

# ----------------------------
# Build box plot
# ----------------------------
fig = px.box(
    df_box,
    x=LOT_COL,
    y=METRIC_COL,
    color="Instrument",
    boxmode="group",       # side-by-side boxes per lot
    points="outliers",     # cleaner; change to "all" if N is small
)

fig.update_layout(
    title="Mean Read Length (bp) by Cleaning_Solution_Lot (colored by Instrument)",
    xaxis_title="Cleaning_Solution_Lot",
    yaxis_title="Mean Read Length (bp)",
    legend_title="Instrument",
    height=550
)

# Apply ordered lots
fig.update_xaxes(
    categoryorder="array",
    categoryarray=lot_order
)

fig.show()


In [ ]:
import plotly.express as px

# ----------------------------
# CONFIG
# ----------------------------
LOT_COL = "ExT_Sequencing_Reagent_Lot"
METRIC_COL = "Mean_Read_Length_bp"

# ----------------------------
# Prepare data (minimal, safe filtering)
# ----------------------------
df_box = df_hampel_len[
    df_hampel_len[LOT_COL].notna() &
    df_hampel_len[METRIC_COL].notna() &
    df_hampel_len["Instrument"].notna()
].copy()

# Ensure lot numbers are treated as categorical (not numeric sorting)
df_box[LOT_COL] = df_box[LOT_COL].astype(str)

# ----------------------------
# Order chip lots by median read length (VERY helpful visually)
# ----------------------------
lot_order = (
    df_box.groupby(LOT_COL)[METRIC_COL]
    .median()
    .sort_values()
    .index
)

# ----------------------------
# Build box plot
# ----------------------------
fig = px.box(
    df_box,
    x=LOT_COL,
    y=METRIC_COL,
    color="Instrument",
    boxmode="group",       # side-by-side boxes per lot
    points="outliers",     # cleaner; change to "all" if N is small
)

fig.update_layout(
    title="Mean Read Length (bp) by ExT_Sequencing_Reagent Lot Number (colored by Instrument)",
    xaxis_title="ExT_Sequencing_Reagent_Lot",
    yaxis_title="Mean Read Length (bp)",
    legend_title="Instrument",
    height=550
)

# Apply ordered lots
fig.update_xaxes(
    categoryorder="array",
    categoryarray=lot_order
)

fig.show()


In [ ]:
import plotly.express as px

# ----------------------------
# CONFIG
# ----------------------------
LOT_COL = "ExT_Wash_Solution_Lot"
METRIC_COL = "Mean_Read_Length_bp"

# ----------------------------
# Prepare data (minimal, safe filtering)
# ----------------------------
df_box = df_hampel_len[
    df_hampel_len[LOT_COL].notna() &
    df_hampel_len[METRIC_COL].notna() &
    df_hampel_len["Instrument"].notna()
].copy()

# Ensure lot numbers are treated as categorical (not numeric sorting)
df_box[LOT_COL] = df_box[LOT_COL].astype(str)

# ----------------------------
# Order chip lots by median read length (VERY helpful visually)
# ----------------------------
lot_order = (
    df_box.groupby(LOT_COL)[METRIC_COL]
    .median()
    .sort_values()
    .index
)

# ----------------------------
# Build box plot
# ----------------------------
fig = px.box(
    df_box,
    x=LOT_COL,
    y=METRIC_COL,
    color="Instrument",
    boxmode="group",       # side-by-side boxes per lot
    points="outliers",     # cleaner; change to "all" if N is small
)

fig.update_layout(
    title="Mean Read Length (bp) by ExT_Wash_Solution Lot Number (colored by Instrument)",
    xaxis_title="ExT_Wash_Solution_Lot",
    yaxis_title="Mean Read Length (bp)",
    legend_title="Instrument",
    height=550
)

# Apply ordered lots
fig.update_xaxes(
    categoryorder="array",
    categoryarray=lot_order
)

fig.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_metrics["Mean_Read_Length_bp"], kde=True)
plt.title("Distribution of Mean Read Length")
plt.xlabel("Mean Read Length (bp)")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_metrics["Usable_Reads_%"], kde=True)
plt.title("Distribution of Usable_Reads_%")
plt.xlabel("% Usable Reads")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(
    data=df_metrics_sorted,
    x="Run_Date",
    y="ISP_Loading_%",
    marker="o"
)
plt.title("ISP Loading (%) Over Time")
plt.xlabel("Run Date")
plt.ylabel("ISP Loading (%)")
plt.xticks(rotation=45)
plt.show()


In [ ]:
  plt.figure(figsize=(10,5))
sns.lineplot(
    data=df_metrics_sorted,
    x="Run_Date",
    y="Mean_Read_Length_bp",
    marker="o"
)
plt.title("Mean Read Length Over Time")
plt.xlabel("Run Date")
plt.ylabel("Mean Read Length (bp)")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(
    data=df_metrics,
    x="Chip_Lot_Number",
    y="ISP_Loading_%"
)
plt.title("ISP Loading (%) by Chip Lot")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(
    data=df_metrics,
    x="ExT_Sequencing_Reagent_Lot",
    y="Mean_Read_Length_bp"
)
plt.title("Read Length by ExT Sequencing Reagent Lot")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Remove fields that should NOT be part of QC correlations
cols_to_exclude = ["Addressable_Wells", "Chip_Type"]

# Create a filtered numeric dataframe
df_corr = df_metrics.drop(columns=cols_to_exclude, errors="ignore")

plt.figure(figsize=(10,8))
corr = df_corr.corr(numeric_only=True)

sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of NGS Metrics (Filtered)")
plt.show()


In [ ]:


# Sort by run date
df_metrics = df_metrics.sort_values("Run_Date")

# Create Levey–Jennings style plot
plt.figure(figsize=(12, 6))

plt.plot(
    df_metrics["Run_Date"],
    df_metrics["Mean_Read_Length_bp"],
    marker="o",
    label="Mean Read Length (bp)"
)

plt.plot(
    df_metrics["Run_Date"],
    df_metrics["Median_Read_Length_bp"],
    marker="o",
    label="Median Read Length (bp)"
)

plt.plot(
    df_metrics["Run_Date"],
    df_metrics["Mode_bp"],
    marker="o",
    label="Mode (bp)"
)

plt.xlabel("Run Date")
plt.ylabel("Read Length (bp)")
plt.title("Levey–Jennings Plot: Read Length Metrics Over Time")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Ensure Instrument column exists
if "Instrument" not in df_metrics.columns:
    print("Instrument column missing!")
else:
    # Melt the dataframe for easy grouped plotting
    df_long = df_metrics.melt(
        id_vars=["Instrument"],
        value_vars=["Mean_Read_Length_bp", "Median_Read_Length_bp", "Mode_bp"],
        var_name="Metric",
        value_name="Read_Length"
    )

    # Create the grouped comparison plot
    plt.figure(figsize=(12,6))
    sns.boxplot(
        data=df_long,
        x="Instrument",
        y="Read_Length",
        hue="Metric"
    )

    plt.title("Comparison of Read Length Metrics Across Instruments")
    plt.xlabel("Instrument")
    plt.ylabel("Read Length (bp)")
    plt.xticks(rotation=45)
    plt.legend(title="Metric")
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ----------------------------
# Box plot: Total Reads by Instrument (fixed Y scale)
# ----------------------------

if "Instrument" not in df_metrics.columns:
    print("Instrument column missing!")
elif "Total_Reads" not in df_metrics.columns:
    print("Total_Reads column missing!")
else:
    df_metrics["Total_Reads"] = pd.to_numeric(df_metrics["Total_Reads"], errors="coerce")

    plt.figure(figsize=(12, 6))
    sns.boxplot(
        data=df_metrics,
        x="Instrument",
        y="Total_Reads"
    )

    plt.title("Comparison of Total Reads Across Instruments")
    plt.xlabel("Instrument")
    plt.ylabel("Total Reads")
    plt.xticks(rotation=45)

    # 🔑 FIX: lock Y-axis to a meaningful range (adjust if needed)
    plt.ylim(1.5e7, 2.6e7)

    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df_metrics, x="Instrument", y="Mean_Read_Length_bp")
plt.title("Read Length by Instrument")
plt.show()


In [ ]:
sns.scatterplot(
    data=df_metrics,
    x="ISP_Loading_%",
    y="Total_Reads",
    hue="Instrument",
    s=100
)
plt.title("ISP Loading vs Total Reads")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=df_metrics,
    x="ISP_Loading_%",
    y="Usable_Reads_%",
    hue="Instrument",
    s=120
)
plt.title("ISP Loading % vs Usable Reads %")
plt.xlabel("ISP Loading (%)")
plt.ylabel("Usable Reads (%)")
plt.grid(True, alpha=0.3)
plt.show()



# Statistical Analysis of differences between instruments


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu

# Keep valid values only
df_test = df_metrics[
    df_metrics["Instrument"].notna() &
    df_metrics["Total_Reads"].notna()
].copy()


In [ ]:
insts = df_test["Instrument"].unique()
assert len(insts) == 2, "This test is for exactly 2 instruments"

x = df_test[df_test["Instrument"] == insts[0]]["Total_Reads"]
y = df_test[df_test["Instrument"] == insts[1]]["Total_Reads"]

u_stat, p_value = mannwhitneyu(x, y, alternative="two-sided")

print(f"Mann–Whitney U test")
print(f"Instruments: {insts[0]} vs {insts[1]}")
print(f"U statistic = {u_stat:.2f}")
print(f"p-value = {p_value:.4g}")


| p-value       | Interpretation                             |
| ------------- | ------------------------------------------ |
| **p ≥ 0.05**  | No statistically significant difference    |
| **p < 0.05**  | Statistically significant difference       |
| **p < 0.01**  | Strong evidence of difference              |
| **p < 0.001** | Very strong evidence (systemic difference) |
